# Tokenization Explained
Tokenization is the process of splitting text into smaller units called **tokens** before feeding it to a language model.

A token is not always a full word — it can be a word, part of a word, punctuation, or even a single character.

In [11]:
pip install tiktoken transformers

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## 1. Simple Word Tokenization (no library)
The most basic form — just split on spaces.

In [12]:
text = "The quick brown fox jumps over the lazy dog."

tokens = text.split()
print("Tokens:", tokens)
print("Total tokens:", len(tokens))

Tokens: ['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog.']
Total tokens: 9


## 2. What LLMs Actually Use — BPE (Byte Pair Encoding)
Modern LLMs like GPT use **BPE tokenization** via the `tiktoken` library.

Words get split into subword pieces — this helps the model handle rare or unknown words.

In [13]:
import tiktoken

# cl100k_base is used by GPT-4 and GPT-3.5
enc = tiktoken.get_encoding('cl100k_base')

text = "The quick brown fox jumps over the lazy dog."

token_ids = enc.encode(text)
print("Token IDs:", token_ids)
print("Total tokens:", len(token_ids))

Token IDs: [791, 4062, 14198, 39935, 35308, 927, 279, 16053, 5679, 13]
Total tokens: 10


## 3. Decode Token IDs Back to Text

In [14]:
# Decode all tokens back to the original text
decoded = enc.decode(token_ids)
print("Decoded text:", decoded)

# See each token individually
print("\nToken breakdown:")
for token_id in token_ids:
    print(f"  ID {token_id:6} -> '{enc.decode([token_id])}'")

Decoded text: The quick brown fox jumps over the lazy dog.

Token breakdown:
  ID    791 -> 'The'
  ID   4062 -> ' quick'
  ID  14198 -> ' brown'
  ID  39935 -> ' fox'
  ID  35308 -> ' jumps'
  ID    927 -> ' over'
  ID    279 -> ' the'
  ID  16053 -> ' lazy'
  ID   5679 -> ' dog'
  ID     13 -> '.'


## 4. Tokens Are Not Always Full Words
Let's see how an unusual word gets broken into subword tokens.

In [15]:
words = ["cat", "cats", "tokenization", "unhappiness", "ChatGPT", "supercalifragilistic"]

print(f"{'Word':<25} {'Tokens':<10} {'Token pieces'}")
print("-" * 60)
for word in words:
    ids = enc.encode(word)
    pieces = [enc.decode([i]) for i in ids]
    print(f"{word:<25} {len(ids):<10} {pieces}")

Word                      Tokens     Token pieces
------------------------------------------------------------
cat                       1          ['cat']
cats                      1          ['cats']
tokenization              2          ['token', 'ization']
unhappiness               3          ['un', 'h', 'appiness']
ChatGPT                   3          ['Chat', 'G', 'PT']
supercalifragilistic      7          ['sup', 'erc', 'al', 'if', 'rag', 'il', 'istic']


## 5. Token Count Affects Cost and Context Window
LLMs have a **context window** (max tokens they can process at once).
API pricing is also based on token count — so fewer tokens = cheaper.

In [16]:
def count_tokens(text):
    return len(enc.encode(text))

short_text = "Hello!"
long_text  = "Artificial intelligence is transforming every industry by enabling machines to learn from data and make decisions."

print(f"Short text  : {count_tokens(short_text):>4} tokens -> '{short_text}'")
print(f"Long text   : {count_tokens(long_text):>4} tokens -> '{long_text[:50]}...'") 

Short text  :    2 tokens -> 'Hello!'
Long text   :   18 tokens -> 'Artificial intelligence is transforming every indu...'


## 6. Tokenization in Hugging Face (BERT style)
BERT-based models use **WordPiece** tokenization — similar idea but different vocabulary.

In [17]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

text = "Tokenization splits text into smaller pieces."
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print("Tokens  :", tokens)
print("IDs     :", token_ids)
print("Decoded :", tokenizer.decode(token_ids))

Tokens  : ['token', '##ization', 'splits', 'text', 'into', 'smaller', 'pieces', '.']
IDs     : [101, 19204, 3989, 19584, 3793, 2046, 3760, 4109, 1012, 102]
Decoded : [CLS] tokenization splits text into smaller pieces. [SEP]


## Summary

| Concept | Description |
|---|---|
| Token | Smallest unit of text a model processes |
| Token ID | Integer that maps to a token in the vocabulary |
| BPE | Byte Pair Encoding — used by GPT models |
| WordPiece | Tokenization used by BERT models |
| Context window | Max number of tokens a model can handle at once |
| Subword token | A token that is part of a word (e.g. `un`, `happy`, `ness`) |